# Data-driven optimal weights (K=3 / K=4 / K=5)

Fit **static** GMM on the training window **1971-03-31 → 1990-12-31** only, then solve
long-only mean-variance portfolios on the **three investable sleeves** for each regime.

- Regime detection uses all **17 factors** (same feature matrix as the GMM pipeline).
- Portfolio weights are optimized on **SPXT / LUATTRUU / BCOMTR** only.
- Weights are **frozen** after training and reused in backtests from 1990 onward.

**No look-ahead:** training ends 1990-12-31; post-1990 regime labels come from the
existing walk-forward CSVs (`walk_forward_k3/4/5.csv`).

In [ ]:
import sys
from pathlib import Path

import pandas as pd


def _find_pmr_root() -> Path:
    start = Path.cwd().resolve()
    for parent in [start, *start.parents]:
        if (parent / "scripts" / "paths.py").is_file():
            return parent
    raise FileNotFoundError("Run from inside pmr_paper/")


PROJECT_ROOT = _find_pmr_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.paths import OUTPUT_DIR, load_features
from scripts.portfolio_allocation import (
    DEFAULT_TRAIN_END,
    DEFAULT_TRAIN_START,
    build_regime_portfolios,
    save_regime_portfolios,
)

features = load_features()
print(f"Features: {features.shape} ({features.index.min().date()} → {features.index.max().date()})")
print(f"Training window: {DEFAULT_TRAIN_START} → {DEFAULT_TRAIN_END}")

## Fit static GMM + MV weights per regime

For each K ∈ {3, 4, 5}:
1. Fit static GMM on 1971–1990 training slice (17 factors).
2. Map sklearn components → economic regime ids (Hungarian labeling).
3. Extract μₖ, Σₖ per regime; subset to 3 investable assets.
4. Solve long-only max-Sharpe: wₖ* with w ≥ 0, Σw = 1.
5. Save frozen weights to `data/outputs/data_driven_portfolios_k{k}.csv`.

In [ ]:
PORTFOLIO_TABLES = {}
SAVED_PATHS = {}

for k in (3, 4, 5):
    table = build_regime_portfolios(features, k)
    path = save_regime_portfolios(table, k)
    PORTFOLIO_TABLES[k] = table
    SAVED_PATHS[k] = path
    print(f"\n=== K={k} — saved to {path} ===")
    display(table.round(3))

## Extended CORE6 portfolios (six investable TR indices)

Same training window and GMM-on-17-factors setup, but MV optimization uses:
**SPXT, LUATTRUU, LF98TRUU, BCOMTR, MXEF, BCIT1T**.

Saved to `data/outputs/data_driven_portfolios_k{k}_core6.csv`.

In [ ]:
from scripts.portfolio_allocation import CORE6_COLS, save_regime_portfolios

CORE6_TABLES = {}
CORE6_SAVED_PATHS = {}

for k in (3, 4, 5):
    table = build_regime_portfolios(features, k, investable_cols=CORE6_COLS)
    path = save_regime_portfolios(table, k, variant="core6")
    CORE6_TABLES[k] = table
    CORE6_SAVED_PATHS[k] = path
    print(f"\n=== K={k} CORE6 — saved to {path} ===")
    display(table.round(3))

## Summary

These CSVs are consumed by `data_driven_3`, `data_driven_4`, and `data_driven_5` in
`01_strategy_comparison.ipynb`. CORE6 variants (`data_driven_3/4/5_core6`) use the
`*_core6.csv` files. Each strategy switches to the frozen regime portfolio using
walk-forward hard labels (or probability blends) for the matching K.